# 02 — Momentum Signals

Analysis of cross-sectional momentum signals. Tests multiple lookback periods and visualizes signal IC decay.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Auto-generate synthetic data if none exists
from statarb.utils import load_config
from statarb.data.storage import DataStore
from experiments.generate_synthetic_data import generate_all

cfg = load_config("../experiments/config.yaml")
data_cfg = cfg.get("data", {})
syn_cfg = cfg.get("synthetic", {})
exchange = data_cfg.get("exchange", "binance")
base_dir = data_cfg.get("base_dir", "../data")

store = DataStore(base_dir=base_dir)
symbols = store.list_symbols(exchange, "1d")
if not symbols:
    print("Generating synthetic data...")
    generate_all(
        n_assets=syn_cfg.get("n_assets", 20),
        n_days=syn_cfg.get("n_days", 1000),
        start_date=syn_cfg.get("start_date", "2021-01-01"),
        seed=syn_cfg.get("seed", 42),
        exchange=exchange,
        base_dir=base_dir,
    )
    symbols = store.list_symbols(exchange, "1d")

prices  = store.build_panel(symbols, exchange, "1d", field="close")
volume  = store.build_volume_panel(symbols, exchange, "1d")

from statarb.data.features import FeatureEngine
fe = FeatureEngine()
returns = fe.log_returns(prices)
vol_ratio = fe.volume_ma_ratio(volume, window=21)

print(f"Loaded: {len(symbols)} assets x {len(prices)} days")
print(f"Date range: {prices.index[0].date()} → {prices.index[-1].date()}")


## Signal Construction

In [ ]:
from statarb.signals.momentum import MomentumSignals
mom = MomentumSignals()

# Build a panel of momentum signals at different lookbacks
signals = {
    "ts_mom_7d":   mom.time_series_momentum(returns, lookback=7),
    "ts_mom_21d":  mom.time_series_momentum(returns, lookback=21),
    "ts_mom_63d":  mom.time_series_momentum(returns, lookback=63),
    "ts_mom_126d": mom.time_series_momentum(returns, lookback=126),
    "price_mom_6_1": mom.momentum_6_1(returns),
    "sharpe_mom_63d": mom.sharpe_momentum(returns, lookback=63),
    "ma_cross_20_60": mom.moving_average_crossover(prices, fast=20, slow=60),
}

# Cross-sectional rank each signal to [-1, 1]
ranked = {name: fe.cross_sectional_rank(sig) for name, sig in signals.items()}

# Signal coverage (fraction of non-NaN at each date)
coverage = {name: sig.notna().mean(axis=1) for name, sig in ranked.items()}
print("Signal coverage (mean fraction of assets with valid signal):")
for name, cov in coverage.items():
    print(f"  {name:25s}: {cov.mean():.1%}")


## Signal Decay — IC by Forward Horizon

In [ ]:
from experiments._utils import signal_decay_ic

max_horizon = 20
decay = {}
for name in ["ts_mom_7d", "ts_mom_21d", "ts_mom_63d", "ts_mom_126d"]:
    decay[name] = signal_decay_ic(ranked[name], returns, max_horizon=max_horizon)

fig, ax = plt.subplots(figsize=(10, 4))
for name, ic_series in decay.items():
    ax.plot(ic_series.index, ic_series.values, marker="o", markersize=4,
            linewidth=1.5, label=name)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Forward Horizon (days)")
ax.set_ylabel("Rank IC (Spearman)")
ax.set_title("Momentum Signal IC Decay by Lookback", fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## Backtest Performance by Lookback

In [ ]:
from experiments._utils import backtest_signals
from statarb.backtest.execution import ExecutionModel

em = ExecutionModel(market_order_cost=0.0020)
results = backtest_signals(ranked, returns, em, periods_per_year=365)
print(results[["sharpe", "gross_sharpe", "cost_drag", "annualized_return",
               "max_drawdown", "avg_turnover"]].round(3).to_string())


In [ ]:
# Best momentum signal cumulative return
best = results["sharpe"].idxmax()
from experiments._utils import run_bt
net_rets, result_dict = run_bt(ranked[best], returns, em)
cum = (1 + net_rets.fillna(0)).cumprod() - 1

fig, axes = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={"height_ratios": [3, 1]})
cum.plot(ax=axes[0])
axes[0].set_title(f"Best Momentum Signal: {best}", fontweight="bold")
axes[0].set_ylabel("Cumulative Return")
axes[0].axhline(0, color="black", linewidth=0.6)

# Drawdown
dd = (1 + net_rets.fillna(0)).cumprod()
dd = (dd - dd.cummax()) / dd.cummax()
dd.plot(ax=axes[1], color="crimson")
axes[1].fill_between(dd.index, dd.values, 0, color="crimson", alpha=0.3)
axes[1].set_ylabel("Drawdown")
plt.tight_layout()
plt.show()

from statarb.evaluation.metrics import PerformanceMetrics
pm = PerformanceMetrics()
print(f"Sharpe:        {pm.sharpe_ratio(net_rets, periods_per_year=365):.2f}")
print(f"Ann. Return:   {pm.annualized_return(net_rets)*100:.1f}%")
print(f"Max Drawdown:  {pm.max_drawdown(net_rets)*100:.1f}%")


## Volume-Conditioned Momentum

In [ ]:
vol_cond = fe.cross_sectional_rank(
    mom.volume_weighted_momentum(returns, vol_ratio, lookback=63)
)
base = ranked["ts_mom_63d"]

# IC on high-volume vs low-volume days
activity_ratio = vol_ratio.mean(axis=1)
high_vol_mask = activity_ratio > activity_ratio.rolling(21).mean()

fwd1 = returns.shift(-1)
def daily_ic(sig, fwd, dates):
    ics = []
    for d in dates:
        if d not in sig.index or d not in fwd.index:
            ics.append(np.nan)
            continue
        s = sig.loc[d].dropna()
        r = fwd.loc[d].reindex(s.index).dropna()
        s2 = s.reindex(r.index)
        if len(s2) < 5:
            ics.append(np.nan)
        else:
            ics.append(float(s2.corr(r, method="spearman")))
    return pd.Series(ics, index=dates)

common_dates = base.index.intersection(fwd1.index)
ic = daily_ic(base, fwd1, common_dates)
hv_mask = high_vol_mask.reindex(common_dates).fillna(False)

print(f"Base TS Momentum IC (high-vol days): {ic[hv_mask].mean():.4f}")
print(f"Base TS Momentum IC (low-vol days):  {ic[~hv_mask].mean():.4f}")
print("=> Higher IC on high-volume days supports informed trading hypothesis")
